# 프로토콜 v1.1 — E2c 채점: bayesrul 직접 실행 결과(HNN, MCD, DE)와 PI-BDP의 짝지은 비교 (탐색 분석, CPU)

E2c bayesrul 실행 노트북이 저장한 창별 예측분포(평균, 표준편차)와 PI-BDP의 고정 예측을 **같은 시험 창**에서 비교합니다.

- **창과 레이블 (v1.1 4.1절):** bayesrul의 0.1 Hz 창(길이 30, 간격 10)과 건전 구간 상한 레이블(엔진별 상한 = 건전 구간 마지막의 RUL). 창의 사이클은 창 마지막 행의 사이클입니다.
- **PI-BDP (4.2절):** 사이클 t의 창에는 사이클 t − 1 뒤의 출력에서 1을 뺀 값(분위수 모두)을 씁니다. 보정한 90% 구간은 원고 식 (12)와 고정된 c₉₀입니다. 각 엔진의 첫 사이클 창은 **모든 모델에서** 뺍니다. 모든 예측값·분위수에 [0, 엔진별 상한]을 똑같이 적용합니다(E2c CruiseBench 채점과 같은 방식).
- **비교 모델 점 예측 (4.2절):** HNN·MCD는 예측분포 평균의 시드 평균, DE는 앙상블 5개 평균의 평균.
- **확률 지표 (5절):** 90% 포함률, 평균 폭, interval score(α = 0.1), 분위수 9개({0.025, …, 0.975}) pinball loss 평균(CRPS 근사). 가우시안 출력은 같은 분위수를 계산해 씁니다. 여러 시드·앙상블이 있는 방법은 **창마다 시드(앙상블)별 점수를 평균**합니다(한 번 학습한 모델의 기대 성능). 이 집계 방식은 프로토콜에 적혀 있지 않아 실행 전에 이렇게 정했습니다.
- **검정 (6절):** PI-BDP V0 − 각 모델의 (1) 통합 RMSE 차이, (2) 평균 interval score 차이(PI-BDP는 보정한 구간)를 계층 붓스트랩(서브셋 → 엔진, 100,000회)으로. 95% 신뢰구간과 보정하지 않은 p값.
- **서술 (9절):** 신뢰구간이 비교 모델 쪽으로 0을 벗어나면 "덜 정확", 0을 포함하면 "비슷", PI-BDP 쪽으로 벗어나면 "더 정확". RMSE와 interval score에 따로 적용합니다.
- **원 논문 지표:** NLL, RMSCE, sharpness 등은 실행 노트북이 원 코드(`compute_metrics`)로 전체 시험 창에서 계산한 값을 그대로 옮깁니다.
- **보조 (4.3절):** 상한 없는 사이클 단위 RMSE(기술 통계).

**실행 전:** `RUN_COMMIT_URL`에 이 노트북을 커밋한 주소를 넣으세요. 런타임은 CPU면 됩니다.

In [ ]:
import os, sys, json, glob, time, hashlib, re
import numpy as np, pandas as pd
from scipy.stats import norm
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
IN3 = f'{ROOT}/phase3'; RUN = f'{ROOT}/addendum_v11/e2c_bayesrul'; OUT = f'{ROOT}/addendum_v11'
RUN_COMMIT_URL = ''
assert re.match(r'https://github\.com/.+/commit/[0-9a-f]{7,40}$', RUN_COMMIT_URL), '이 노트북을 먼저 커밋하고 주소를 넣으세요.'
SUBSETS = ['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008']
QL = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]; QC = [f'q{int(round(q * 1000)):03d}' for q in QL]
K = ['subset', 'unit', 'cycle']
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
# 입력이 고정 당시·실행 당시와 같은지 확인
REF3 = json.load(open(f'{IN3}/sha256_phase3.json'))
bad = [k for k, v in REF3.items() if os.path.exists(f'{IN3}/{k}') and sha(f'{IN3}/{k}') != v]; assert not bad, f'Phase 3 파일이 바뀜: {bad}'
REFR = json.load(open(f'{RUN}/sha256_e2c_bayesrul.json'))
bad = [k for k, v in REFR['files'].items() if (k.startswith('preds/') or k.startswith('windows_')) and sha(f'{RUN}/{k}') != v]
assert not bad, f'실행 결과 파일이 바뀜: {bad}'
dec = json.load(open(f'{RUN}/br_compute_decision.json'))
c90 = json.load(open(f'{ROOT}/phase1/calibration.json'))['c']['90']; EPS = 0.5
print('Phase 3 해시', len(REF3), '| 실행 해시 확인 | bayesrul', REFR['bayesrul'][:7], '| 실행 수', {k: dec[k] for k in ['n_hnn', 'n_mcd', 'n_rep_hnn']}, '| c90', c90)

In [ ]:
# === 창, 레이블 확인, 같은 창에 모든 모델 놓기 ===
W = pd.read_csv(f'{RUN}/windows_main.csv.gz'); W['subset'] = W.file.str.replace('N-CMAPSS_', '', regex=False)
V0 = pd.concat([pd.read_csv(f'{IN3}/test_PI-BDP_V0_{s}.csv.gz') for s in SUBSETS], ignore_index=True)
eol = (V0.cycle + V0.true).groupby([V0.subset, V0.unit]).max().rename('eol').reset_index()
W = W.merge(eol, on=['subset', 'unit'], how='left'); assert W.eol.notna().all(), '시험 엔진이 맞지 않음'
assert np.allclose(W.rul_raw, W.eol - W.cycle), '원 RUL이 EOL - cycle과 맞지 않음'
W['cap'] = W.groupby(['subset', 'unit']).label.transform('max')
assert np.allclose(W.label, np.minimum(W.rul_raw, W.cap)), '건전 구간 상한 레이블이 min(RUL, 엔진별 상한)과 맞지 않음'
first = W.cycle == W.groupby(['subset', 'unit']).cycle.transform('min'); n_first = int(first.sum())
prev = V0.assign(cycle=V0.cycle + 1)[K + QC]
B = W[~first].merge(prev, on=K, how='left'); assert B[QC].notna().all().all(), 't-1 PI-BDP 출력이 없는 창이 있음'
assert (B.idx.values == np.sort(B.idx.values)).all()
y = B.label.values; cap = B.cap.values; IDX = B.idx.values
clip = lambda v: np.clip(v, 0, cap[:, None] if np.ndim(v) == 2 else cap)
print('시험 창', len(W), '| 채점 창', len(B), '| 뺀 첫 사이클 창', n_first, '| 엔진', B.groupby(['subset', 'unit']).ngroups)

In [ ]:
# === 모델별 창 단위 예측 (점, 90% 구간, 분위수 9개) ===
def pinball(Q, y):   # Q: (n, 9)
    d = y[:, None] - Q; tau = np.array(QL)[None, :]
    return np.mean(np.maximum(tau * d, (tau - 1) * d), axis=1)
def iscore(L, U, y): return (U - L) + 20 * (L - y) * (y < L) + 20 * (y - U) * (y > U)
def scores(L, U, Q):
    return dict(cover=((y >= L) & (y <= U)).astype(float), width=U - L, iscore=iscore(L, U, y), pinball=pinball(Q, y))

Qp = clip(B[QC].values - 1); m = B.q500.values - 1                      # PI-BDP: t-1 출력 - 1
Lc = np.maximum(m - c90 * np.maximum(m - (B.q050.values - 1), EPS), 0); Uc = m + c90 * np.maximum((B.q950.values - 1) - m, EPS)
PT = {'PI-BDP V0': clip(m)}
PR = {'PI-BDP V0': scores(clip(Lc), clip(Uc), Qp), 'PI-BDP V0 (raw quantiles)': scores(Qp[:, 1], Qp[:, 7], Qp)}

def load(method):
    fs = sorted(glob.glob(f'{RUN}/preds/main_{method}_[0-9][0-9][0-9].csv.gz')); out = []
    for f in fs:
        d = pd.read_csv(f); assert np.allclose(d.labels.values, W.label.values), f'{f}: 레이블 순서가 창과 다름'
        out.append(d.set_index('idx').loc[IDX])
    return out
RUNS = {m: load(m) for m in ['HNN', 'MCD', 'DE']}
assert len(RUNS['HNN']) == dec['n_hnn'] and len(RUNS['MCD']) == dec['n_mcd'] and len(RUNS['DE']) == 5, {k: len(v) for k, v in RUNS.items()}
z = norm.ppf(QL)
PER_RUN = []
for meth, runs in RUNS.items():
    PT[meth] = clip(np.mean([d.preds.values for d in runs], axis=0))   # 시드(앙상블) 평균의 평균
    acc = None
    for k, d in enumerate(runs):
        mu, sd = d.preds.values, d.stds.values; Q = clip(mu[:, None] + sd[:, None] * z[None, :])
        s = scores(Q[:, 1], Q[:, 7], Q)
        acc = s if acc is None else {kk: acc[kk] + s[kk] for kk in s}
        PER_RUN.append(dict(method=meth, run=k, rmse=float(np.sqrt(np.mean((clip(mu) - y) ** 2))), **{kk: float(v.mean()) for kk, v in s.items()}))
    PR[meth] = {kk: v / len(runs) for kk, v in acc.items()}               # 창마다 시드(앙상블)별 점수의 평균
PER_RUN = pd.DataFrame(PER_RUN)

In [ ]:
# === 지표 표 ===
def rmse(e): return float(np.sqrt(np.mean(np.square(e))))
rows = []
for name in ['PI-BDP V0', 'HNN', 'MCD', 'DE']:
    e = PT[name] - y; st = pd.Series(e).groupby(B.subset.values).apply(rmse)
    rows.append(dict(model=name, pooled_rmse=rmse(e), mean_subset_rmse=float(st.mean()), **{s: float(st[s]) for s in SUBSETS}))
POINT = pd.DataFrame(rows)
PROB = pd.DataFrame([dict(model=k, cov90=v['cover'].mean(), width90=v['width'].mean(), iscore90=v['iscore'].mean(), pinball9=v['pinball'].mean())
                     for k, v in PR.items()])
PROB_SUB = pd.DataFrame([dict(model=k, subset=s, cov90=v['cover'][B.subset.values == s].mean(), iscore90=v['iscore'][B.subset.values == s].mean())
                         for k, v in PR.items() for s in SUBSETS])
POINT.to_csv(f'{OUT}/e2c_br_point.csv', index=False); PROB.to_csv(f'{OUT}/e2c_br_prob.csv', index=False)
PROB_SUB.to_csv(f'{OUT}/e2c_br_prob_subset.csv', index=False); PER_RUN.to_csv(f'{OUT}/e2c_br_per_run.csv', index=False)
print(POINT.round(3).to_string(index=False)); print(); print(PROB.round(3).to_string(index=False))
print(); print(PER_RUN.groupby('method')[['rmse', 'cover', 'iscore', 'pinball']].agg(['mean', 'std']).round(3).to_string())

In [ ]:
# === 계층 붓스트랩 (서브셋 → 엔진), PI-BDP V0 - 각 모델 ===
def hier_boot(ea, eb, subset, unit, kind, n_boot=100000, seed=1):
    # kind='rmse': ea, eb는 오차, 통계량은 통합 RMSE 차이. kind='mean': ea, eb는 창별 점수, 통계량은 평균 차이.
    va, vb = (ea ** 2, eb ** 2) if kind == 'rmse' else (ea, eb)
    U = pd.DataFrame({'subset': subset, 'unit': unit, 'a': va, 'b': vb}).groupby(['subset', 'unit']).agg(
        sa=('a', 'sum'), sb=('b', 'sum'), n=('a', 'size')).reset_index()
    subs = sorted(U.subset.unique()); rng = np.random.default_rng(seed)
    per = [U[U.subset == s][['sa', 'sb', 'n']].values for s in subs]
    SA = np.zeros(n_boot); SB = np.zeros(n_boot); N = np.zeros(n_boot)
    pick = rng.integers(0, len(subs), (n_boot, len(subs)))
    for j in range(len(subs)):
        for si, arr in enumerate(per):
            r = np.flatnonzero(pick[:, j] == si)
            if not len(r): continue
            tot = arr[rng.integers(0, len(arr), (len(r), len(arr)))].sum(1)
            SA[r] += tot[:, 0]; SB[r] += tot[:, 1]; N[r] += tot[:, 2]
    f = (lambda a, n: np.sqrt(a / n)) if kind == 'rmse' else (lambda a, n: a / n)
    d = f(SA, N) - f(SB, N); obs = float(f(U.sa.sum(), U.n.sum()) - f(U.sb.sum(), U.n.sum()))
    return dict(diff=obs, ci_lo=float(np.percentile(d, 2.5)), ci_hi=float(np.percentile(d, 97.5)),
                p_boot=float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))))
def wording(r):
    if r['ci_lo'] > 0: return 'PI-BDP less accurate'
    if r['ci_hi'] < 0: return 'PI-BDP more accurate (exploratory)'
    return 'comparable'
tests = []
for mdl in ['HNN', 'MCD', 'DE']:
    r = hier_boot(PT['PI-BDP V0'] - y, PT[mdl] - y, B.subset.values, B.unit.values, 'rmse')
    tests.append(dict(metric='pooled RMSE', comparison=f'V0 - {mdl}', **r, wording=wording(r)))
    r = hier_boot(PR['PI-BDP V0']['iscore'], PR[mdl]['iscore'], B.subset.values, B.unit.values, 'mean')
    tests.append(dict(metric='interval score (90%)', comparison=f'V0 (calibrated) - {mdl}', **r, wording=wording(r)))
TESTS = pd.DataFrame(tests); TESTS.to_csv(f'{OUT}/e2c_br_tests.csv', index=False)
print(TESTS.round(4).to_string(index=False))

In [ ]:
# === 원 논문 지표 (실행 노트북이 원 코드로 계산), 재현 확인, 보조 채점 ===
ORIG = pd.read_csv(f'{RUN}/e2c_br_original_metrics_main.csv')
ORIG_AGG = ORIG.groupby('method')[['mae', 'rmse', 'nll', 'rmsce', 'sharp']].agg(['mean', 'std'])
ORIG_AGG.columns = ['_'.join(x) for x in ORIG_AGG.columns]; ORIG_AGG = ORIG_AGG.reset_index(); ORIG_AGG.to_csv(f'{OUT}/e2c_br_original_metrics.csv', index=False)
REP = pd.read_csv(f'{RUN}/e2c_br_replication.csv')
cyc = V0[K + ['true', 'q500']].rename(columns={'q500': 'PI-BDP V0'})
for mdl in ['HNN', 'MCD', 'DE']:
    full = np.mean([pd.read_csv(f).preds.values for f in sorted(glob.glob(f'{RUN}/preds/main_{mdl}_[0-9][0-9][0-9].csv.gz'))], axis=0)
    cm = W[K].assign(p=full).groupby(K, as_index=False).p.mean().rename(columns={'p': mdl})
    cyc = cyc.merge(cm, on=K, how='inner')
SEC = pd.DataFrame([dict(model=mm, cycle_rmse_uncapped=rmse(cyc[mm] - cyc.true), n_cycles=len(cyc)) for mm in ['PI-BDP V0', 'HNN', 'MCD', 'DE']])
SEC.to_csv(f'{OUT}/e2c_br_cycle_level.csv', index=False)
print(ORIG_AGG.round(3).to_string(index=False)); print(); print(REP.T.to_string()); print(); print(SEC.round(3).to_string(index=False))

In [ ]:
# === 요약 + 해시 ===
txt = '\n'.join([f"Addendum v1.1 E2c bayesrul scoring (exploratory) {time.strftime('%Y-%m-%d %H:%M:%S')}  run={RUN_COMMIT_URL}",
    f"bayesrul {REFR['bayesrul'][:7]}; runs: HNN {dec['n_hnn']}, MCD {dec['n_mcd']}, DE 5 (from HNN); windows scored {len(B)}; first-cycle windows excluded {n_first}",
    '', 'Replication check (DS01-DS05 pooled, HNN, all test windows):', REP.T.to_string(),
    '', 'Point predictions (healthy-phase capped labels, 0.1 Hz windows):', POINT.round(3).to_string(index=False),
    '', f'Probabilistic (PI-BDP calibrated with c90={c90}; multi-run methods averaged per window over runs):', PROB.round(3).to_string(index=False),
    '', 'Paired hierarchical bootstrap V0 - model, 100,000 resamples:', TESTS.round(4).to_string(index=False),
    '', 'Original bayesrul metrics (compute_metrics, all test windows, mean/std over runs):', ORIG_AGG.round(3).to_string(index=False),
    '', 'Secondary, cycle-level uncapped (descriptive):', SEC.round(3).to_string(index=False)])
open(f'{OUT}/e2c_br_summary.txt', 'w').write(txt)
H = {os.path.basename(p): sha(p) for p in sorted(glob.glob(f'{OUT}/e2c_br_*.*'))}
json.dump(H, open(f'{OUT}/sha256_e2c_br_score.json', 'w'), indent=1); print(txt)